# COMP64702 - RAG Culinary Assistant
## Notebook 02: Benchmark Dataset Creation (Auto-Generated via LLM)
### =============================================================================
#### Strategy: Feed chunks of your corpus into an LLM (Qwen) and ask it to
#### generate QA pairs automatically. 

## Pipeline:
####   1. Sample documents from corpus
####   2. For each doc, prompt Qwen to generate QA pairs of different types
####   3. Parse and validate the generated pairs
####   4. Deduplicate and quality-filter
####   5. Split into train/test and save
### =============================================================================
 

In [ ]:
# Imports and setup
 
import json
import os
import glob
import random
import re
import time
from datetime import datetime
from collections import Counter
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
 
os.chdir(r"D:\text mining\rag project")
print(f"Working directory: {os.getcwd()}")
 
os.makedirs("data/benchmark", exist_ok=True)
 
# Load HF token and login
load_dotenv()
login(token=os.getenv("HF_TOKEN"))
print("Logged in to HuggingFace")
 
random.seed(42)
 

Working directory: D:\text mining\rag project


Logged in to HuggingFace


In [ ]:
# Load the LLM (Qwen2.5-0.5B-Instruct)
# We use the same model required for generation so no extra downloads needed
 
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
 
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()
 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model loaded on: {device}")

Loading Qwen/Qwen2.5-0.5B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded on: cpu


In [ ]:
# Helper: run a prompt through Qwen
 
def run_qwen(system_prompt, user_prompt, max_new_tokens=512, temperature=0.7):
    """
    Runs a prompt through Qwen2.5-0.5B-Instruct using the chat template.
    Returns the generated text string.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
 
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
 
    inputs = tokenizer([text], return_tensors="pt").to(device)
 
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
 
    # Decode only the newly generated tokens
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()
 
 
# Quick test
test_out = run_qwen("You are a helpful assistant.", "Say hello in one word.")
print(f"Model test output: '{test_out}'")

Model test output: 'Hello!'


In [ ]:
# Load corpus documents
 
def load_corpus():
    docs = []
    patterns = {
        "wikipedia": "data/raw/wikipedia_articles/*.json",
        "wikibooks": "data/raw/wikibooks_recipes/*.json",
        "blog":      "data/raw/blog_posts/*.json",
    }
    for source, pattern in patterns.items():
        for fp in glob.glob(pattern):
            with open(fp, encoding="utf-8") as f:
                doc = json.load(f)
            doc["source_type"] = source
            docs.append(doc)
    print(f"Loaded {len(docs)} documents")
    return docs
 
corpus = load_corpus()
 
# Filter to only docs with enough content (min 300 words)
rich_docs = [d for d in corpus if d.get("word_count", 0) >= 300]
print(f"Rich documents (300+ words): {len(rich_docs)}")
 

Loaded 237 documents
Rich documents (300+ words): 193


In [ ]:
# QA generation prompts per question type
 
# System prompt used for all QA generation
SYSTEM_PROMPT = """You are an expert at creating question-answer pairs for evaluating
retrieval-augmented generation (RAG) systems in the domain of East Asian cuisine.
Your task is to generate high-quality QA pairs from the provided text.
 
Rules:
- Questions must be answerable ONLY from the provided text
- Answers must be factually correct and directly supported by the text
- Answers should be 1-4 sentences, concise but complete
- Do NOT make up information not in the text
- Output ONLY valid JSON, nothing else"""
 
 
# One prompt template per question type
QA_PROMPTS = {
 
    "factual": """Given the following text about East Asian cuisine, generate {n} factual question-answer pairs.
Factual questions ask about specific facts: ingredients, origins, names, definitions.
 
Text:
{text}
 
Output a JSON array like this (output ONLY the JSON, no other text):
[
  {{"question": "...", "answer": "...", "difficulty": "easy/medium/hard"}},
  ...
]""",
 
    "procedural": """Given the following text about East Asian cuisine, generate {n} procedural question-answer pairs.
Procedural questions ask HOW something is made or done (methods, steps, techniques).
 
Text:
{text}
 
Output a JSON array (output ONLY the JSON, no other text):
[
  {{"question": "...", "answer": "...", "difficulty": "easy/medium/hard"}},
  ...
]""",
 
    "ingredient": """Given the following text about East Asian cuisine, generate {n} ingredient question-answer pairs.
Ingredient questions ask WHAT goes into a dish (components, seasonings, substitutions).
 
Text:
{text}
 
Output a JSON array (output ONLY the JSON, no other text):
[
  {{"question": "...", "answer": "...", "difficulty": "easy/medium/hard"}},
  ...
]""",
 
    "cultural": """Given the following text about East Asian cuisine, generate {n} cultural question-answer pairs.
Cultural questions ask about history, tradition, significance, or cultural context.
 
Text:
{text}
 
Output a JSON array (output ONLY the JSON, no other text):
[
  {{"question": "...", "answer": "...", "difficulty": "easy/medium/hard"}},
  ...
]""",
}
 

In [ ]:
# Parse LLM output into QA pairs
 
def parse_qa_output(raw_output, question_type, source_title):
    """
    Parses the LLM's JSON output into a list of QA pair dicts.
    Handles malformed JSON gracefully.
    """
    # Try to extract a JSON array from the output
    # Sometimes the model adds preamble text before the JSON
    json_match = re.search(r'\[.*\]', raw_output, re.DOTALL)
    if not json_match:
        return []
 
    try:
        pairs = json.loads(json_match.group())
    except json.JSONDecodeError:
        # Try to fix common issues: trailing commas, single quotes
        cleaned = json_match.group()
        cleaned = re.sub(r',\s*\]', ']', cleaned)   # trailing comma in array
        cleaned = re.sub(r',\s*\}', '}', cleaned)   # trailing comma in object
        try:
            pairs = json.loads(cleaned)
        except Exception:
            return []
 
    validated = []
    for p in pairs:
        if not isinstance(p, dict):
            continue
        q = p.get("question", "").strip()
        a = p.get("answer",   "").strip()
        d = p.get("difficulty", "medium").strip().lower()
 
        # Basic quality checks
        if len(q) < 10 or len(a) < 20:
            continue
        if not q.endswith("?"):
            q = q + "?"
        if d not in ["easy", "medium", "hard"]:
            d = "medium"
 
        validated.append({
            "question":     q,
            "answer":       a,
            "type":         question_type,
            "difficulty":   d,
            "source_title": source_title,
        })
 
    return validated
 

In [ ]:
# Generate QA pairs from corpus
#
# Strategy:
#   - Sample docs from corpus weighted toward richer documents
#   - For each sampled doc, generate pairs for each question type
#   - Target: ~100-120 total QA pairs
 

N_PER_TYPE = 2

SAMPLE_CONFIG = {
    "wikipedia": 20,   # 20 docs × 4 types × 2 pairs = 160 attempts → ~100 valid
    "wikibooks": 8,    # 8 docs  × 4 types × 2 pairs = 64  attempts → ~40  valid
    "blog":      5,    # 5 docs  × 4 types × 2 pairs = 40  attempts → ~25  valid
}
 
# Sample documents
sampled_docs = []
for source_type, n in SAMPLE_CONFIG.items():
    pool = [d for d in rich_docs if d["source_type"] == source_type]
    # Weight toward longer docs — more content = better QA generation
    weights  = [d.get("word_count", 300) for d in pool]
    selected = random.choices(pool, weights=weights, k=min(n, len(pool)))
    sampled_docs.extend(selected)
    print(f"Sampled {len(selected)} {source_type} docs")
 
print(f"\nTotal docs to process: {len(sampled_docs)}")
print(f"Estimated QA pairs (if all succeed): ~{len(sampled_docs) * 4 * N_PER_TYPE}")
 
 
def truncate_text(text, max_words=400):
    """
    Truncate text to max_words for the prompt.
    Qwen 0.5B has a limited context window so we keep chunks short.
    We take the first 400 words which usually covers the most informative content.
    """
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words]) + "..."
 

Sampled 20 wikipedia docs
Sampled 8 wikibooks docs
Sampled 5 blog docs

Total docs to process: 33
Estimated QA pairs (if all succeed): ~264


In [ ]:
# Main generation loop
 
all_qa_pairs  = []
total_attempts = 0
total_parsed   = 0
failed_count   = 0
 
print("\nStarting QA generation...\n")
print(f"{'Doc':<5} {'Title':<40} {'Type':<12} {'Generated':<10}")
print("─" * 72)
 
for doc_idx, doc in enumerate(sampled_docs, 1):
    title   = doc.get("title", "Unknown")
    text    = truncate_text(doc.get("text", ""), max_words=400)
    source  = doc.get("source_type", "unknown")
 
    if len(text.split()) < 100:
        print(f"[{doc_idx:02d}] Skipping (too short): {title}")
        continue
 
    for q_type, prompt_template in QA_PROMPTS.items():
        total_attempts += 1
 
        user_prompt = prompt_template.format(n=N_PER_TYPE, text=text)
 
        try:
            raw_output = run_qwen(
                SYSTEM_PROMPT,
                user_prompt,
                max_new_tokens=400,
                temperature=0.7,
            )
 
            pairs = parse_qa_output(raw_output, q_type, title)
 
            # Add unique IDs
            for pair in pairs:
                pair["id"] = f"{q_type[0].upper()}{len(all_qa_pairs)+1:03d}"
                pair["source_doc_type"] = source
                all_qa_pairs.append(pair)
                total_parsed += 1
 
            status = f"{len(pairs)} pairs"
            print(f"[{doc_idx:02d}] {title[:38]:<40} {q_type:<12} {status}")
 
        except Exception as e:
            failed_count += 1
            print(f"[{doc_idx:02d}] {title[:38]:<40} {q_type:<12} FAILED: {type(e).__name__}")
 
        # Small delay to avoid memory pressure
        time.sleep(0.5)
 
print(f"\n{'─'*72}")
print(f"Generation complete")
print(f"  Attempts  : {total_attempts}")
print(f"  QA pairs  : {total_parsed}")
print(f"  Failed    : {failed_count}")
 


Starting QA generation...

Doc   Title                                    Type         Generated 
────────────────────────────────────────────────────────────────────────
[01] Sake                                     factual      0 pairs
[01] Sake                                     procedural   2 pairs
[01] Sake                                     ingredient   2 pairs
[01] Sake                                     cultural     2 pairs
[02] Beijing cuisine                          factual      2 pairs
[02] Beijing cuisine                          procedural   2 pairs
[02] Beijing cuisine                          ingredient   2 pairs
[02] Beijing cuisine                          cultural     1 pairs
[03] Glutinous rice                           factual      2 pairs
[03] Glutinous rice                           procedural   2 pairs
[03] Glutinous rice                           ingredient   2 pairs
[03] Glutinous rice                           cultural     2 pairs
[04] Dorayaki           

In [ ]:
# Deduplicate and quality filter
 
def deduplicate(qa_pairs, similarity_threshold=0.8):
    """
    Remove duplicate or near-duplicate questions.
    Uses simple word overlap (Jaccard similarity) — fast and good enough.
    """
    seen_questions = []
    unique_pairs   = []
 
    for pair in qa_pairs:
        q_words = set(pair["question"].lower().split())
        is_dup  = False
 
        for seen_q in seen_questions:
            # Jaccard similarity
            intersection = q_words & seen_q
            union        = q_words | seen_q
            similarity   = len(intersection) / len(union) if union else 0
 
            if similarity >= similarity_threshold:
                is_dup = True
                break
 
        if not is_dup:
            seen_questions.append(q_words)
            unique_pairs.append(pair)
 
    return unique_pairs
 
 
def quality_filter(qa_pairs, min_answer_words=15, min_question_words=5):
    """Filter out very short or low-quality QA pairs."""
    filtered = []
    for pair in qa_pairs:
        q_words = len(pair["question"].split())
        a_words = len(pair["answer"].split())
 
        if q_words < min_question_words:
            continue
        if a_words < min_answer_words:
            continue
        # Skip pairs where answer is suspiciously similar to question
        q_set = set(pair["question"].lower().split())
        a_set = set(pair["answer"].lower().split())
        overlap = len(q_set & a_set) / max(len(q_set), 1)
        if overlap > 0.7:
            continue
 
        filtered.append(pair)
 
    return filtered
 
 
print(f"Before deduplication : {len(all_qa_pairs)} pairs")
deduped   = deduplicate(all_qa_pairs)
print(f"After deduplication  : {len(deduped)} pairs")
filtered  = quality_filter(deduped)
print(f"After quality filter : {len(filtered)} pairs")
 
# If we have fewer than 50 pairs, lower the thresholds
if len(filtered) < 50:
    print("\nFewer than 50 pairs after filtering — relaxing quality threshold")
    filtered = quality_filter(deduped, min_answer_words=10, min_question_words=4)
    print(f"After relaxed filter : {len(filtered)} pairs")
 

Before deduplication : 243 pairs
After deduplication  : 221 pairs
After quality filter : 111 pairs


In [ ]:
# Add comparative and multi-hop pairs
# These are harder to auto-generate from single docs so we add them manually
 
MANUAL_PAIRS = [
    {
        "id": "COMP001",
        "question": "How does Chinese hot pot differ from Japanese shabu-shabu?",
        "answer": "Chinese hot pot uses richly spiced broths (often split spicy and mild) with diverse ingredients including offal and fish balls. Japanese shabu-shabu uses a delicate kombu broth and focuses on thinly sliced high-quality beef dipped briefly and eaten with ponzu or sesame sauce. Shabu-shabu is more restrained while hot pot is more varied.",
        "type": "comparative",
        "difficulty": "medium",
        "source_title": "Hot pot",
        "source_doc_type": "wikipedia",
    },
    {
        "id": "COMP002",
        "question": "What are the key differences between Chinese, Japanese, and Korean cuisine?",
        "answer": "Chinese cuisine emphasises bold sauces and regional diversity across eight culinary traditions. Japanese cuisine prioritises minimalism, seasonality, and preserving natural flavours. Korean cuisine features bold fermented flavours, heavy use of chilli and garlic, and communal dining with banchan side dishes. All three centre rice and noodles but use distinct seasoning profiles.",
        "type": "comparative",
        "difficulty": "hard",
        "source_title": "East Asian cuisine",
        "source_doc_type": "wikipedia",
    },
    {
        "id": "COMP003",
        "question": "How does Cantonese cuisine differ from Sichuan cuisine?",
        "answer": "Cantonese cuisine prioritises freshness and subtlety with light sauces to enhance natural flavours. Sichuan cuisine is defined by the bold mala flavour combination of Sichuan peppercorn and dried chilli, creating intense numbing heat. Cantonese cooking favours steaming and light stir-frying while Sichuan uses braising and heavy seasoning.",
        "type": "comparative",
        "difficulty": "medium",
        "source_title": "Cantonese cuisine",
        "source_doc_type": "wikipedia",
    },
    {
        "id": "MULTI001",
        "question": "How are fermentation techniques used across East Asian cuisines?",
        "answer": "Fermentation underpins all three major East Asian cuisines. Korean cuisine uses it for kimchi, doenjang, and gochujang. Japanese cuisine relies on it for miso, sake, soy sauce, and natto. Chinese cuisine uses it for doubanjiang, black bean paste, and Shaoxing wine. All three traditions developed fermentation as a preservation technique now prized for umami flavour.",
        "type": "multi-hop",
        "difficulty": "hard",
        "source_title": "Fermentation in food processing",
        "source_doc_type": "wikipedia",
    },
    {
        "id": "MULTI002",
        "question": "How is tofu used differently across Chinese, Japanese, and Korean cuisines?",
        "answer": "In Chinese cooking, firm tofu is braised or cooked in spicy sauces like mapo tofu. In Japanese cuisine, silken tofu is served cold or in miso soup, while firm tofu appears in agedashi and hot pot. In Korean cuisine, soft tofu features in sundubu-jjigae stew and pan-fried dubu-jorim. China and Korea favour firmer varieties in cooked dishes while Japan often uses silken tofu.",
        "type": "multi-hop",
        "difficulty": "hard",
        "source_title": "Tofu",
        "source_doc_type": "wikipedia",
    },
]
 
for pair in MANUAL_PAIRS:
    filtered.append(pair)
 
print(f"\nAfter adding manual comparative/multi-hop pairs: {len(filtered)} total")
 


After adding manual comparative/multi-hop pairs: 116 total


In [ ]:
# Reassign clean IDs, shuffle, and split
 
# Reassign clean sequential IDs
type_counters = Counter()
for pair in filtered:
    t = pair["type"]
    type_counters[t] += 1
    prefix_map = {
        "factual": "F", "procedural": "P", "ingredient": "I",
        "cultural": "CU", "comparative": "C", "multi-hop": "M",
    }
    prefix      = prefix_map.get(t, "Q")
    pair["id"]  = f"{prefix}{type_counters[t]:03d}"
 
# Shuffle
random.shuffle(filtered)
 
# 80/20 train/test split
split     = int(len(filtered) * 0.8)
train_set = filtered[:split]
test_set  = filtered[split:]
 
# Test queries only (no answers — matches demo day format)
test_queries_only = [
    {"id": q["id"], "question": q["question"]}
    for q in test_set
]
 
print(f"\nFinal dataset:")
print(f"  Total pairs : {len(filtered)}")
print(f"  Train       : {len(train_set)}")
print(f"  Test        : {len(test_set)}")
print(f"\nBy type:")
for t, n in sorted(Counter(q['type'] for q in filtered).items()):
    print(f"  {t:<15}: {n}")
print(f"\nBy difficulty:")
for d, n in sorted(Counter(q['difficulty'] for q in filtered).items()):
    print(f"  {d:<10}: {n}")
 


Final dataset:
  Total pairs : 116
  Train       : 92
  Test        : 24

By type:
  comparative    : 3
  cultural       : 29
  factual        : 26
  ingredient     : 25
  multi-hop      : 2
  procedural     : 31

By difficulty:
  easy      : 33
  hard      : 42
  medium    : 41


In [ ]:
# Save all benchmark files
 
def save_json(filepath, data):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  Saved: {filepath}")
 
 
full_benchmark = {
    "created_at":              datetime.now().isoformat(),
    "cuisine":                 "East Asian",
    "generation_method":       "LLM-assisted (Qwen2.5-0.5B-Instruct) + manual comparative/multi-hop pairs",
    "total_pairs":             len(filtered),
    "train_pairs":             len(train_set),
    "test_pairs":              len(test_set),
    "type_distribution":       dict(Counter(q["type"] for q in filtered)),
    "difficulty_distribution": dict(Counter(q["difficulty"] for q in filtered)),
    "qa_pairs":                filtered,
}
 
print("\nSaving benchmark files...")
save_json("data/benchmark/benchmark_full.json",          full_benchmark)
save_json("data/benchmark/train_set.json",               train_set)
save_json("data/benchmark/test_set_with_answers.json",   test_set)
save_json("data/benchmark/test_set_queries_only.json",   {"queries": test_queries_only})
 
print(f"\nBenchmark creation complete.")
print(f"Total QA pairs: {len(filtered)}")
print(f"Proceed to 03_ingestion.ipynb")


Saving benchmark files...
  Saved: data/benchmark/benchmark_full.json
  Saved: data/benchmark/train_set.json
  Saved: data/benchmark/test_set_with_answers.json
  Saved: data/benchmark/test_set_queries_only.json

Benchmark creation complete.
Total QA pairs: 116
Proceed to 03_ingestion.ipynb
